In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the feature dataset
features_file = "../data/processed/features.csv"

df = pd.read_csv(features_file)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (3116, 13)

Columns:
['location_id', 'latitude', 'longitude', 'dist_road_m', 'dist_hospital_m', 'dist_school_m', 'dist_police_m', 'dist_market_m', 'flood_risk', 'land_use_class', 'dist_landuse_commercial_m', 'population_density', 'elevation_m']


,location_id,latitude,longitude,dist_road_m,dist_hospital_m,dist_school_m,dist_police_m,dist_market_m,flood_risk,land_use_class,dist_landuse_commercial_m,population_density,elevation_m
0,0,18.893956,72.776333,3056.443643,6089.817763,188107.179606,189143.480646,189032.898599,0.0,forest,3537.854912,NaN,NaN
1,1,18.898956,72.776333,2959.537695,5569.615011,187555.975554,188591.953759,188481.425349,0.0,forest,3403.468513,NaN,NaN
2,2,18.903956,72.776333,2879.412938,5056.510336,187004.785265,188040.438676,187929.964222,0.0,forest,3355.940510,NaN,NaN
3,3,18.908956,72.776333,2899.102096,4552.904296,186453.608865,187488.935503,187378.515328,0.0,forest,3390.053600,NaN,NaN
4,4,18.913956,72.776333,2996.117293,4062.331171,185902.446480,186937.444348,186827.078779,0.0,residential,3512.142991,NaN,NaN


In [3]:
# Calculate distance percentiles
road_p25 = df["dist_road_m"].quantile(0.25)
road_p50 = df["dist_road_m"].quantile(0.50)
road_p75 = df["dist_road_m"].quantile(0.75)

hospital_p25 = df["dist_hospital_m"].quantile(0.25)
hospital_p50 = df["dist_hospital_m"].quantile(0.50)
hospital_p75 = df["dist_hospital_m"].quantile(0.75)

print("Road distance thresholds:")
print("25th percentile:", road_p25)
print("50th percentile:", road_p50)
print("75th percentile:", road_p75)

print("\nHospital distance thresholds:")
print("25th percentile:", hospital_p25)
print("50th percentile:", hospital_p50)
print("75th percentile:", hospital_p75)

Road distance thresholds:
25th percentile: 28.3585221097488
50th percentile: 358.6077565161703
75th percentile: 2087.9509044119854

Hospital distance thresholds:
25th percentile: 635.919820758241
50th percentile: 1950.958923403607
75th percentile: 4008.68943140008


In [4]:
# Define labeling thresholds
road_good_threshold = road_p50
hospital_good_threshold = hospital_p50

road_bad_threshold = road_p75
hospital_bad_threshold = hospital_p75

# Start with all locations unlabelled
df["label"] = np.nan

# Good Site = close to road AND close to hospital AND flood risk is low
good_condition = (
    (df["dist_road_m"] <= road_good_threshold) &
    (df["dist_hospital_m"] <= hospital_good_threshold) &
    (df["flood_risk"] == 0)
)

# Poor Site = very far from road OR very far from hospital
poor_condition = (
    (df["dist_road_m"] >= road_bad_threshold) |
    (df["dist_hospital_m"] >= hospital_bad_threshold)
)

# Assign labels
df.loc[good_condition, "label"] = 1
df.loc[poor_condition, "label"] = 0

print("Good sites:", (df["label"] == 1).sum())
print("Poor sites:", (df["label"] == 0).sum())
print("Unlabelled:", df["label"].isna().sum())

Good sites: 1303
Poor sites: 945
Unlabelled: 868


In [5]:
# Keep only locations with a valid label
labelled_df = df.dropna(subset=["label"]).copy()

# Convert label to integer
labelled_df["label"] = labelled_df["label"].astype(int)

print("Final labelled dataset shape:", labelled_df.shape)

print("\nLabel distribution:")
print(labelled_df["label"].value_counts().sort_index())

print("\nLabel meaning:")
print("0 = Poor Site")
print("1 = Good Site")

Final labelled dataset shape: (2248, 14)

Label distribution:
label
0     945
1    1303
Name: count, dtype: int64

Label meaning:
0 = Poor Site
1 = Good Site


In [6]:
# Save the labelled dataset
output_file = "../data/processed/labelled_sites.csv"

labelled_df.to_csv(output_file, index=False)

print("Labelled dataset saved successfully!")
print(output_file)

Labelled dataset saved successfully!
../data/processed/labelled_sites.csv


In [7]:
# Verify the saved dataset
check_df = pd.read_csv(output_file)

print("Saved dataset shape:", check_df.shape)
print("\nFirst 5 rows:")
display(check_df.head())

print("\nLabel counts:")
print(check_df["label"].value_counts().sort_index())

Saved dataset shape: (2248, 14)

First 5 rows:


,location_id,latitude,longitude,dist_road_m,dist_hospital_m,dist_school_m,dist_police_m,dist_market_m,flood_risk,land_use_class,dist_landuse_commercial_m,population_density,elevation_m,label
0,0,18.893956,72.776333,3056.443643,6089.817763,188107.179606,189143.480646,189032.898599,0.0,forest,3537.854912,NaN,NaN,0
1,1,18.898956,72.776333,2959.537695,5569.615011,187555.975554,188591.953759,188481.425349,0.0,forest,3403.468513,NaN,NaN,0
2,2,18.903956,72.776333,2879.412938,5056.510336,187004.785265,188040.438676,187929.964222,0.0,forest,3355.940510,NaN,NaN,0
3,3,18.908956,72.776333,2899.102096,4552.904296,186453.608865,187488.935503,187378.515328,0.0,forest,3390.053600,NaN,NaN,0
4,4,18.913956,72.776333,2996.117293,4062.331171,185902.446480,186937.444348,186827.078779,0.0,residential,3512.142991,NaN,NaN,0



Label counts:
label
0     945
1    1303
Name: count, dtype: int64
